# 13.4 跨模态 (Cross-Modal)

跨模态技术解决不同模态之间的对齐和融合问题。

本节涵盖：模态对齐、模态融合、统一模型

## 1. 模态对齐与融合

**模态对齐**：将不同模态的表征映射到共享空间。
- **CLIP**：对比学习对齐图像和文本
- **对齐损失**：InfoNCE / 对比损失

**模态融合**：将多模态信息合并为统一表征。
- **早期融合**：在输入层拼接不同模态
- **中期融合**：在中间层通过交叉注意力融合
- **晚期融合**：分别处理各模态，最后合并结果

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class CLIPStyleAlignment(nn.Module):
    def __init__(self, d_image=64, d_text=64, d_shared=32):
        super().__init__()
        self.image_proj = nn.Linear(d_image, d_shared)
        self.text_proj = nn.Linear(d_text, d_shared)
        self.logit_scale = nn.Parameter(torch.ones(1) * math.log(1 / 0.07))

    def forward(self, image_features, text_features):
        image_embeds = F.normalize(self.image_proj(image_features), dim=-1)
        text_embeds = F.normalize(self.text_proj(text_features), dim=-1)
        logit_scale = self.logit_scale.exp().clamp(max=100)
        logits = logit_scale * image_embeds @ text_embeds.T
        labels = torch.arange(len(logits), device=logits.device)
        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.T, labels)
        return (loss_i2t + loss_t2i) / 2, logits

clip = CLIPStyleAlignment()
img_features = torch.randn(4, 64)
txt_features = torch.randn(4, 64)
loss, sim_matrix = clip(img_features, txt_features)

print('=== Modality Alignment (CLIP-style) ===')
print(f'Image features: {img_features.shape}')
print(f'Text features: {txt_features.shape}')
print(f'Contrastive loss: {loss.item():.4f}')
print(f'\nSimilarity matrix:')
print(f'  {sim_matrix.detach().softmax(dim=-1).tolist()}')
print(f'\nKey: Contrastive learning aligns different modalities in shared space.')
print(f'Diagonal should be high (matching pairs), off-diagonal low.')
print(f'\n=== Unified Models ===')
print(f'GPT-4o: Native multimodal, processes all modalities in one model')
print(f'Gemini: Natively multimodal from training, not stitched together')
print(f'\nKey: Unified models process all modalities natively,')
print(f'achieving better cross-modal understanding than separate models.')